# Task 4 — TF-IDF Vectorization and Similarity Analysis

**Objective:** convert the cleaned `combined_text` field to TF-IDF vectors and use cosine similarity to find related TED Talks.

A TF-IDF row represents one talk; a column represents a vocabulary feature (word or word pair); a non-zero value represents the importance of that feature in that talk relative to the corpus.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from pathlib import Path
import pandas as pd


def find_project_root():
    """Find the project folder from the current Jupyter working directory."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "ted_talks_merged.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw/ted_talks_merged.csv. "
        "Open Jupyter from the TED_Talks_Recommendation_Ajay project folder."
    )


PROJECT_DIR = find_project_root()
RAW_DATA_PATH = PROJECT_DIR / "data" / "raw" / "ted_talks_merged.csv"
PROCESSED_DATA_PATH = PROJECT_DIR / "data" / "processed" / "ted_talks_cleaned.csv"
print("Project folder:", PROJECT_DIR)

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError("Run Task 3 first to create data/processed/ted_talks_cleaned.csv")

df = pd.read_csv(PROCESSED_DATA_PATH)
required = ["title", "combined_text"]
missing_columns = [column for column in required if column not in df.columns]
if missing_columns:
    raise ValueError(f"Cleaned dataset is missing required columns: {missing_columns}")

df["combined_text"] = df["combined_text"].fillna("").astype(str)
usable_mask = df["combined_text"].str.strip().ne("")
work_df = df.loc[usable_mask].reset_index(drop=True)
print("Total talks:", len(df))
print("Talks with usable combined text:", len(work_df))

Project folder: C:\Users\Ajay\Desktop\TED_Talks_Recommendation_Ajay
Total talks: 2550
Talks with usable combined text: 2550


## 1. Create TF-IDF vectors

`min_df=2` ignores terms that occur in only one talk. `ngram_range=(1, 2)` includes single words and two-word phrases. The matrix stays sparse to reduce memory use.

In [3]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
)
tfidf_matrix = vectorizer.fit_transform(work_df["combined_text"])
feature_names = vectorizer.get_feature_names_out()

print("TF-IDF matrix shape (talks, features):", tfidf_matrix.shape)
print("Number of vocabulary terms:", len(feature_names))
print("First 30 vocabulary terms:")
print(feature_names[:30].tolist())
print("Non-zero matrix values:", tfidf_matrix.nnz)
print("Matrix density:", round(tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]), 6))

TF-IDF matrix shape (talks, features): (2550, 20000)
Number of vocabulary terms: 20000
First 30 vocabulary terms:
['00', '000', '000 000', '000 children', '000 days', '000 different', '000 dollar', '000 dollars', '000 employees', '000 feet', '000 genes', '000 hours', '000 kilometers', '000 meters', '000 miles', '000 new', '000 people', '000 pounds', '000 species', '000 square', '000 students', '000 times', '000 tons', '000 year', '000 years', '10', '10 000', '10 11', '10 15', '10 20']
Non-zero matrix values: 1324489
Matrix density: 0.02597


## 2. Calculate similarity for a selected talk

Cosine similarity compares the angle between two TF-IDF vectors. A score closer to 1 means the vectors point in a more similar direction; it is not a guarantee that a person will enjoy the recommended talk.

The selected talk is removed from the result so it cannot recommend itself. We calculate similarity for one query row at a time instead of creating a large dense all-talks matrix.

In [4]:
def recommend_from_index(index, top_n=5):
    if index < 0 or index >= len(work_df):
        raise IndexError("Talk index is outside the dataset.")
    scores = cosine_similarity(tfidf_matrix[index], tfidf_matrix).ravel()
    scores[index] = -1  # exclude the selected talk itself
    best_indices = np.argsort(scores)[::-1][:top_n]
    result = work_df.iloc[best_indices].copy()
    result["similarity_score"] = scores[best_indices]
    return result[[column for column in ["title", "main_speaker", "similarity_score", "url"] if column in result.columns]]

sample_index = 0
print("Selected TED Talk:", work_df.loc[sample_index, "title"])
display(recommend_from_index(sample_index, top_n=5))

Selected TED Talk: Do schools kill creativity?


,title,main_speaker,similarity_score,url
1502,How to escape education's death valley,Ken Robinson,0.364297,https://www.ted.com/talks/ken_robinson_how_to_...
692,Bring on the learning revolution!,Ken Robinson,0.356238,https://www.ted.com/talks/sir_ken_robinson_bri...
21,Nerdcore comedy,Ze Frank,0.268711,https://www.ted.com/talks/ze_frank_s_nerdcore_...
1913,How to run a company with (almost) no rules,Ricardo Semler,0.265060,https://www.ted.com/talks/ricardo_semler_how_t...
2049,How I stopped the Taliban from shutting down m...,Sakena Yacoobi,0.264126,https://www.ted.com/talks/sakena_yacoobi_how_i...


## 3. Test at least three talks

In [5]:
test_indices = sorted(set([0, min(50, len(work_df)-1), min(200, len(work_df)-1)]))
for index in test_indices:
    print("\nSelected talk:", work_df.loc[index, "title"])
    display(recommend_from_index(index, top_n=5))


Selected talk: Do schools kill creativity?


,title,main_speaker,similarity_score,url
1502,How to escape education's death valley,Ken Robinson,0.364297,https://www.ted.com/talks/ken_robinson_how_to_...
692,Bring on the learning revolution!,Ken Robinson,0.356238,https://www.ted.com/talks/sir_ken_robinson_bri...
21,Nerdcore comedy,Ze Frank,0.268711,https://www.ted.com/talks/ze_frank_s_nerdcore_...
1913,How to run a company with (almost) no rules,Ricardo Semler,0.265060,https://www.ted.com/talks/ricardo_semler_how_t...
2049,How I stopped the Taliban from shutting down m...,Sakena Yacoobi,0.264126,https://www.ted.com/talks/sakena_yacoobi_how_i...



Selected talk: The hidden world of shadow cities


,title,main_speaker,similarity_score,url
105,What squatter cities can teach us,Stewart Brand,0.276200,https://www.ted.com/talks/stewart_brand_on_squ...
1855,It’s our city. Let’s fix it,Alessandra Orofino,0.249968,https://www.ted.com/talks/alessandra_orofino_i...
1213,The 4 commandments of cities,Eduardo Paes,0.230347,https://www.ted.com/talks/eduardo_paes_the_4_c...
1603,Ingenious homes in unexpected places,Iwan Baan,0.196747,https://www.ted.com/talks/iwan_baan_ingenious_...
1324,Brilliant designs to fit more people in every ...,Kent Larson,0.195401,https://www.ted.com/talks/kent_larson_brillian...



Selected talk: 5 predictions, from 1984


,title,main_speaker,similarity_score,url
1783,A 30-year history of the future,Nicholas Negroponte,0.284105,https://www.ted.com/talks/nicholas_negroponte_...
285,"The web is more than ""better TV""",Peter Hirshberg,0.242210,https://www.ted.com/talks/peter_hirshberg_on_t...
14,One Laptop per Child,Nicholas Negroponte,0.238568,https://www.ted.com/talks/nicholas_negroponte_...
13,The radical promise of the multi-touch interface,Jeff Han,0.232382,https://www.ted.com/talks/jeff_han_demos_his_b...
370,Design and discovery,David Carson,0.226028,https://www.ted.com/talks/david_carson_on_design


## 4. Observations and limitations

- TF-IDF converts text to a numerical representation that can be compared by a similarity measure.
- Cosine similarity is based on word/phrase overlap and their TF-IDF weights.
- A talk cannot be recommended to itself; its score is explicitly excluded.
- Talks that use different words for the same idea may receive a low score.
- Long transcripts can dominate the text profile, and popular generic words may not distinguish topics well.
- A high similarity score is not the same as a human relevance judgement or a user preference.